# Inspect test-run output — CAM & CLM variable tables

Tabulates the variables actually written by the production `h0` tapes (one CAM test, one CLM test),
tags each by science branch, and checks the **requested whitelist vs what actually landed** in the file.

**Metadata only — this is light.** `xr.open_dataset` is lazy: it reads headers, not data arrays, and
`nbytes` is computed from dtype×shape without a read. Nothing here calls `.values/.load/.compute`,
so file size does not matter. (Equivalent no-Python path: `ncks -m file.nc`.)

Outputs per model: `vars_<model>.csv` and a printed present / missing / unexpected report.


## 1 · Configuration — point at your two test files

In [ ]:
from pathlib import Path
import glob

# EDIT THESE two globs to your test h0 files (run dir or .../archive/<CASE>/atm|lnd/hist/):
CAM_GLOB = "/cluster/work/users/adelez/**/*cam.h0*.nc"
CLM_GLOB = "/cluster/work/users/adelez/**/*clm2.h0*.nc"

OUT_DIR = Path("inspect_output"); OUT_DIR.mkdir(exist_ok=True)
RECORDS_PER_MEMBER = 120   # 10 yr monthly, for size projection

def first_match(pat):
    hits = sorted(glob.glob(pat, recursive=True))
    if not hits: print(f"[!] no file matched {pat}"); return None
    if len(hits) > 1: print(f"[i] {len(hits)} matched {pat}; using {hits[0]}")
    return hits[0]

CAM_FILE = first_match(CAM_GLOB)
CLM_FILE = first_match(CLM_GLOB)
print("CAM:", CAM_FILE); print("CLM:", CLM_FILE)


## 2 · Functions (metadata-only) + the whitelist branch maps

In [ ]:
import numpy as np, pandas as pd, xarray as xr

# dims that are NOT an extra (vertical / pft / column) level:
FLAT_DIMS = {"time","lat","lon","lonu","latu","nbnd","hist_interval","chars","string_length"}

def build_var_table(path):
    """One row per data variable. Lazy open, metadata only — no data is read."""
    ds = xr.open_dataset(path, decode_times=False, decode_cf=False)
    rows=[]
    for name, da in ds.data_vars.items():
        dims=tuple(da.dims); sizes={d:ds.sizes[d] for d in dims}
        extra=[d for d in dims if d not in FLAT_DIMS]           # vertical/pft/etc.
        nontime=[sizes[d] for d in dims if d!="time"]
        per=(int(np.prod(nontime)) if nontime else 1)*da.dtype.itemsize
        rows.append(dict(variable=name, dims=",".join(dims),
            multilevel=bool(extra), level_dim=",".join(extra),
            dtype=str(da.dtype), shape=",".join(str(sizes[d]) for d in dims),
            mib_per_rec=per/1024**2,
            units=da.attrs.get("units",""), long_name=da.attrs.get("long_name","")))
    ds.close()
    return pd.DataFrame(rows).sort_values("variable").reset_index(drop=True)

def tag_and_report(df, branches, model, records=RECORDS_PER_MEMBER):
    """Add a 'branch' column, write CSV, print counts + present/missing/unexpected."""
    v2b = {v:b for b,vs in branches.items() for v in vs}
    df = df.copy()
    df["branch"] = df.variable.map(v2b).fillna("(unexpected)")
    df.to_csv(OUT_DIR/f"vars_{model}.csv", index=False)

    gib = lambda s: s*records/1024
    present = set(df.variable)
    requested = set(v2b)
    missing = sorted(requested - present)          # asked for, not in file
    unexpected = sorted(df.loc[df.branch=="(unexpected)","variable"])  # in file, not requested

    print(f"===== {model.upper()} : {CAM_FILE if model=='cam' else CLM_FILE} =====")
    print(f"variables in file : {len(df)}  (multilevel={int(df.multilevel.sum())}, "
          f"2D={int((~df.multilevel).sum())})")
    print(f"projected size    : ~{gib(df.mib_per_rec.sum()):.2f} GiB/member (uncompressed, {records} recs)")
    print("\nby branch (count | GiB/member):")
    g=(df.assign(g=lambda d:gib(d.mib_per_rec)).groupby("branch")
         .agg(n=("variable","size"), gib=("g","sum")).sort_values("gib",ascending=False))
    print(g.to_string())
    print(f"\nMISSING (requested but absent): {len(missing)}")
    if missing: print("  ", missing)
    print(f"UNEXPECTED (in file, not requested): {len(unexpected)}")
    if unexpected: print("  ", unexpected)
    return df

# ---- whitelist branch maps (from production_diagnostics / clm_diagnostics) ----
CAM_BRANCHES = {
 "radiative": "FSNT FSNTC FLNT FLNTC FLUT FLUTC FSNTOA FSNTOAC SOLIN FSNS FSNSC FLNS FLNSC FSDS FSDSC FLDS SWCF LWCF".split(),
 "ghan_drf": "FSNT_DRF FLNT_DRF FSNTCDRF FLNTCDRF FSDS_DRF FSDSCDRF FSUTADRF FSUS_DRF FLUS".split(),
 "bvoc": "SFISOP SFMTERP SFBCARY cb_ISOP cb_MTERP cb_BCARY MEG_ISOP MEG_MTERP MEG_BCARY emis_ISOP emis_MTERP ISOP MTERP BCARY".split(),
 "soa_core": "SOA_LV SOA_SV H2SO4 SOA_NA SOA_A1 SO4_NA SO4_A1 N_AER cb_SOA_LV cb_SOA_SV cb_H2SO4".split(),
 "npf": "NUCLRATE FORMRATE COAGNUCL GR GRH2SO4 GRSOA ORGNUCL NUCLSOA".split(),
 "tendencies": "SOA_NAcondTend SOA_A1condTend SOA_NAcoagTend SOA_A1coagTend SOA_NA_mixnuc1 SOA_A1_mixnuc1 SO4_NAcondTend SO4_A1condTend SO4_NAcoagTend SO4_A1coagTend SO4_NA_mixnuc1 SO4_A1_mixnuc1".split(),
 "deposition": "SOA_NADDF SOA_A1DDF SO4_NADDF SO4_A1DDF DF_H2SO4 SOA_NASFWET SOA_A1SFWET SO4_NASFWET SO4_A1SFWET WD_A_H2SO4 WD_H2SO4".split(),
 "ccn": "CCN1 CCN2 CCN3 CCN4 CCN5 CCN6 CCN7 CCN_B".split(),
 "optics": "AOD_VIS AEROD_v DOD550 DOD440 DOD870 ABS550 ABS550_A OD550DRY AB550DRY CABS550 A550_BC A550_POM A550_SO4 A550_SS A550_DU".split(),
 "cloud": "CDNUMC TGCLDLWP TGCLDIWP TGCLDCWP CLDTOT CLDLOW CLDMED CLDHGH ACTREL ACTREI ACTNL FCTL FCTI CLOUD CLDLIQ CLDICE AREL AREI AWNC FREQL FREQI NUMLIQ NUMICE".split(),
 "biogeophysical": "TS TREFHT SHFLX LHFLX PRECC PRECL PRECSC PRECSL TAUX TAUY PSL U10 QREFHT LANDFRAC OCNFRAC ICEFRAC SNOWHLND".split(),
 "ch4_ozone": "O3 OH CH4 NO NO2 CO HO2 TROP_P TROP_T TROP_Z".split(),
 "meteorology": "T Q U V OMEGA Z3 PS".split(),
}
CLM_BRANCHES = {
 "radiation_albedo": "FSA FSR FIRA FIRE FSH EFLX_LH_TOT FGR FSDS FLDS FSDSVD FSDSVI FSDSND FSDSNI FSRVD FSRND".split(),
 "snow": "H2OSNO SNOWDP FSNO SNOWLIQ SNOWICE".split(),
 "temperature": "TSA TV TG TSKIN TSOI".split(),
 "vegetation": "TLAI ELAI LAISUN LAISHA TSAI HTOP".split(),
 "megan_drivers": "PARVEGLN BTRANMN".split(),
 "hydrology_et": "QFLX_EVAP_TOT QSOIL QVEGE QVEGT QINTR QOVER QRUNOFF RAIN SNOW H2OSOI SOILLIQ SOILICE ZWT".split(),
 "carbon": "GPP NPP AR HR NEE".split(),
 "other": "WIND PCT_NAT_PFT".split(),
 "megan_emissions": "MEG_isoprene MEG_carene_3 MEG_limonene MEG_myrcene MEG_ocimene_t_b MEG_pinene_a MEG_pinene_b MEG_sabinene".split(),
}


## 3 · CAM test file

In [ ]:
cam = tag_and_report(build_var_table(CAM_FILE), CAM_BRANCHES, "cam") if CAM_FILE else None
cam.head(20) if cam is not None else None


## 4 · CLM test file

In [ ]:
clm = tag_and_report(build_var_table(CLM_FILE), CLM_BRANCHES, "clm") if CLM_FILE else None
clm.head(20) if clm is not None else None


## 5 · How to read the report

- **MISSING** = you requested it in `fincl` but it's not in the file → silently dropped (wrong name) or gated by a flag that's off. Investigate before trusting the run.
- **UNEXPECTED** = in the file but not in your whitelist → usually an automatic coordinate/bounds/scalar (`hyam`, `time_bnds`, `date`…), which is fine. Anything else means the tape wasn't as clean as intended (check `empty_htapes`).
- **multilevel** flags vertical/pft/column fields (CAM `lev/ilev`; CLM `levsoi/levgrnd/levsno/natpft`) — the size drivers.
